# Pathwise — Research tests


## 1. Setup 

In [1]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv(override=True)

from agents import Runner, trace

# My research agents
from pathwiseagenticassistant.agents.research.topic_planning_agent import topic_planning_agent
from pathwiseagenticassistant.agents.research.relevance_judge import relevance_judge_agent, MIN_RELEVANCE_SCORE
from pathwiseagenticassistant.agents.research.critic_agent import critic_agent
from pathwiseagenticassistant.agents.research.research_manager import ResearchManager

# John's agents (search + summarizer)
from pathwiseagenticassistant.tools.web_search_agent import web_search_agent
from pathwiseagenticassistant.tools.summarizer import summarizer_agent

# Budhil's compilation
from pathwiseagenticassistant.agents.compilation.compilation_orchestrator import compile_lesson_stream

# Shared classes
from pathwiseagenticassistant.schemas import LearningRequest, LevelEnum

print("imports OK")

c:\Users\leulg\Desktop\AI-Engineering\pathwiseAgenticAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imports OK


---
# Part A — Quick tests (one agent at a time)

### A1. Topic Planner
Expected: a list of subtopics + ~5 searches with different angles, including one about the learner's tech stack.

In [ ]:
request_text = """
topic: Apache Kafka
level: intermediate
tech_stack: Java, Spring Boot
goal: build event-driven microservices at work
"""

with trace("Pathwise - topic planner test"):
    result = await Runner.run(topic_planning_agent, request_text)

plan = result.final_output
print("Subtopics:", plan.subtopics)
for s in plan.searches:
    print(f"[{s.angle}] {s.query}")
    print(f"     why: {s.reason}")

### A2. Relevance Judge (LLM-as-judge)
Expected: **KEEP** the Spring guide, **DROP** the sales page and the Franz Kafka (writer) article.

In [ ]:
learner = "topic: Apache Kafka, level: intermediate, stack: Java Spring Boot, goal: microservices"

test_articles = [
    "title: Spring for Apache Kafka Reference | url: https://docs.spring.io/spring-kafka/reference/ | snippet: Official guide to producers, consumers and listeners in Spring Boot.",
    "title: Learn Kafka in 7 Days - 80% OFF Today! | url: https://best-course-deals.example.com/kafka | snippet: Buy our bestselling course now, limited offer.",
    "title: The History of Franz Kafka's Novels | url: https://literature.example.com/kafka | snippet: An overview of the Czech writer's major works.",
]

with trace("Pathwise - judge test"):
    for article in test_articles:
        result = await Runner.run(relevance_judge_agent, f"LEARNER: {learner}\nARTICLE: {article}")
        v = result.final_output
        decision = "KEEP" if v.score >= MIN_RELEVANCE_SCORE else "DROP"
        print(f"{decision} | score {v.score} | {v.reason}")

### A3. Critic
Expected: missing = **Consumer groups** (planned, but no article covers it).

In [ ]:
critic_input = """
LEARNER: topic: Apache Kafka, level: intermediate, stack: Java Spring Boot, goal: microservices

PLANNED SUBTOPICS:
- Kafka architecture
- Topics and partitions
- Producers and consumers
- Consumer groups
- Spring Boot integration

ARTICLE NOTES:
1. Title: Kafka Introduction (kafka.apache.org)
   Key points: Kafka is a distributed event log; brokers store topics; topics are split into partitions.
   Subtopics covered: Kafka architecture, topics and partitions

2. Title: Spring for Apache Kafka Reference (docs.spring.io)
   Key points: KafkaTemplate sends messages; @KafkaListener receives them; configure with application.yml.
   Subtopics covered: producers and consumers, Spring Boot integration
"""

with trace("Pathwise - critic test"):
    result = await Runner.run(critic_agent, critic_input)

report = result.final_output
print("Missing:", report.missing_subtopics)
print("Contradictions:", report.contradictions)
print("Follow-up searches:", report.follow_up_queries)

---
# Part B — Full flow: Research → Compilation
Run B1 → B2 → B3 → B4 → B5 in order.

### B1. Learner request 

In [2]:
request = LearningRequest(
    topic="Apache Kafka",
    level=LevelEnum.INTERMEDIATE,
    tech_stack="Java, Spring Boot",
    goal="build event-driven microservices at work",
)

print(request)

topic='Apache Kafka' level=<LevelEnum.INTERMEDIATE: 'intermediate'> tech_stack='Java, Spring Boot' goal='build event-driven microservices at work'


### B2. Run my Research Manager (with John's real agents)
Takes **4–6 minutes** and Expected: up to 6 summaries + a list of gaps.

In [3]:
manager = ResearchManager(web_search_agent, summarizer_agent)
pack = await manager.run(request)

print("Research finished")
print("Number of summaries:", len(pack.summaries))
print("Gaps:", pack.gaps)

Research finished
Number of summaries: 6
Gaps: ['Setting up Kafka with Spring Boot', 'Kafka topics and partitions', 'Integrating Kafka with event-driven architecture', 'Error handling and message acknowledgment in Kafka', 'Monitoring and managing Kafka clusters', 'Best practices for using Kafka in microservices']


### B3. Look inside the research pack
Check: real titles, real URLs, key points about the topic, subtopics.

In [4]:
for i, s in enumerate(pack.summaries, start=1):
    print(f"{i}. {s.title}")
    print(f"   url: {s.url}")
    print(f"   subtopics: {s.subtopics_covered}")
    for point in s.key_points:
        print(f"   - {point}")
    print()

1. Introduction | Apache Kafka
   url: https://kafka.apache.org/20/getting-started/introduction/
   subtopics: ['Topics and Logs', 'Distribution', 'Geo-Replication', 'Producers', 'Consumers', 'Guarantees', 'Kafka as a Messaging System']
   - Kafka is a distributed streaming platform that provides three capabilities: publish/subscribe to streams of records, durable storage of streams, and real-time stream processing.
   - Data in Kafka is organized around topics, each topic has partitioned logs, and each record has a key, value, and timestamp with offsets to identify position; data retention is configurable.
   - There are four core APIs in Kafka: Producer, Consumer, Streams, and Connector, enabling publishing, consuming, processing, and connecting to external systems.
   - The Kafka cluster distributes data across servers via partitions with leader/follower replication for fault tolerance and scalable throughput, and partitions serve as the unit of parallelism.
   - Kafka supports geo-

### B4. Give the pack to Budhil's compilation
Takes a few minutes. Expected last lines: `[done] Lesson verified` (or `returned unverified`) and `Compilation finished: True`.

In [5]:
compiled = None
async for event in compile_lesson_stream(pack):
    print(f"[{event.stage}] {event.message}")
    if event.result is not None:
        compiled = event.result

print("Compilation finished:", compiled is not None)

[planning] Drafting lesson plan
[reviewing] Reviewing lesson plan (round 1)
[planning] Revising lesson plan to fix blocking issues
[reviewing] Reviewing lesson plan (round 2)
[writing] Writing lesson (draft 1)
[verifying] Verifying lesson (draft 1)
[done] Lesson verified
Compilation finished: True
